# Argumind — Fine-Tuning IndoBERT (F-01) & Cross-Encoder NLI (F-02) di Google Colab

Notebook ini menjalankan pelatihan terstruktur sesuai **`research/METHODOLOGY.md`**, **`research/EXPERIMENT_PLAN.md`**, dan **`research/REPRODUCIBILITY.md`**:
1. Menghubungkan Google Colab ke repositori GitHub **`salsabilaputri95/argumind-project`**.
2. Menginstal dependensi Python & memverifikasi integritas checksum `manifest.sha256`.
3. Melatih model **IndoBERT BIO Sequence Labeler** (`F-01`: `Claim`, `ExplicitPremise`, `Evidence`).
4. Melatih **Cross-Encoder NLI Relation Classifier** (`F-02`: `Support`, `Attack`, `NoRelation`) + dekode **Chu-Liu/Edmonds DAG** (`cycle_rate == 0.0`).
5. Mengekspor metrik mentah (`eval_bio.json`, `eval_nli.json`, `metrics_raw.json`) dan bobot model ke Google Drive atau unduhan lokal.

In [ ]:
# @title 1. Cek GPU Google Colab & Kloning Repositori GitHub
!nvidia-smi

import os
from pathlib import Path

REPO_URL = "https://github.com/salsabilaputri95/argumind-project.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
WORK_DIR = Path("/content/argumind-project")

if WORK_DIR.exists():
    %cd /content/argumind-project
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} /content/argumind-project
    %cd /content/argumind-project

!git log -n 1 --oneline

In [ ]:
# @title 2. Instal Dependensi Python & Atur PYTHONPATH Workspace
!pip install -q transformers==4.46.2 datasets==3.1.0 accelerate==1.1.1 \
  networkx==3.4.2 scipy==1.14.1 scikit-learn==1.5.2 pydantic==2.10.1 \
  bert-score==0.3.13 krippendorff==0.6.1 python-docx pymupdf

import sys

import torch

for rel in [
    "argumind-platform/packages/shared-schemas/src",
    "argumind-platform/packages/evaluation-metrics/src",
    "argumind-platform/apps/nlp-worker/src",
    "argumind-platform/apps/api/src",
    ".",
]:
    p = str((WORK_DIR / rel).resolve())
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTHONPATH"] = ":".join(sys.path[:5])
print("PyTorch:", torch.__version__, "| CUDA Available:", torch.cuda.is_available())

In [ ]:
# @title 3. Verifikasi Checksum Dataset (manifest.sha256)
DATA_DIR = "research/data/processed/v1.0"  # @param {type:"string"}
!cd {DATA_DIR} && sha256sum -c manifest.sha256
!wc -l {DATA_DIR}/*.jsonl

In [ ]:
# @title 4. Fine-Tuning IndoBERT BIO Component Extractor (F-01)
MODEL_NAME = "indobenchmark/indobert-base-p2"  # @param ["indobenchmark/indobert-base-p2", "indobenchmark/indobert-large-p2"]
SEED = 42  # @param [42, 100, 2026] {type:"raw"}
EPOCHS = 10  # @param {type:"integer"}
BATCH_SIZE = 16  # @param {type:"integer"}
LR = 2e-5  # @param {type:"number"}

!python -m research.runners.train_indobert_bio \
  --data-dir {DATA_DIR} \
  --output-dir research/outputs/colab_indobert_bio_seed{SEED} \
  --model-name {MODEL_NAME} \
  --seed {SEED} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR}

In [ ]:
# @title 5. Fine-Tuning Cross-Encoder NLI Relation Classifier + DAG Decoder (F-02)
NLI_BASE_MODEL = "indobenchmark/indobert-base-p2"  # @param ["indobenchmark/indobert-base-p2", "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"]
NLI_EPOCHS = 8  # @param {type:"integer"}

!python -m research.runners.train_nli_relations \
  --data-dir {DATA_DIR} \
  --output-dir research/outputs/colab_nli_rel_seed{SEED} \
  --model-name {NLI_BASE_MODEL} \
  --seed {SEED} \
  --epochs {NLI_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LR}

In [ ]:
# @title 6. Jalankan Runner Evaluasi Standar E-01 & E-03b serta Tampilkan Hasil
!python -m research.runners.run_experiment \
  --exp-id E-01 --arch M1_ArgumindHybrid \
  --data-dir {DATA_DIR} --split test --seed {SEED} \
  --bio-model-dir research/outputs/colab_indobert_bio_seed{SEED}/best_model \
  --nli-model-dir research/outputs/colab_nli_rel_seed{SEED}/best_nli_model \
  --output-dir research/outputs/E-01/colab_seed{SEED}

!python -m research.runners.run_experiment \
  --exp-id E-03b --arch M3_GraphGroundedScoring \
  --data-dir {DATA_DIR} --split test --seed {SEED} \
  --output-dir research/outputs/E-03b/colab_seed{SEED}

import json

for path in [
    f"research/outputs/colab_indobert_bio_seed{SEED}/eval_bio.json",
    f"research/outputs/colab_nli_rel_seed{SEED}/eval_nli.json",
    f"research/outputs/E-01/colab_seed{SEED}/metrics_raw.json",
    f"research/outputs/E-03b/colab_seed{SEED}/metrics_raw.json",
]:
    if Path(path).exists():
        print(f"=== {path} ===")
        print(json.dumps(json.loads(Path(path).read_text()), indent=2))

In [ ]:
# @title 7. (Opsional) Simpan Checkpoint & Artefak ke Google Drive / Unduh ZIP
!zip -r -q argumind_colab_outputs_seed{SEED}.zip research/outputs/
print("Arsip siap:", f"/content/argumind-project/argumind_colab_outputs_seed{SEED}.zip")

SAVE_TO_DRIVE = False  # @param {type:"boolean"}
if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/argumind_outputs
    !cp argumind_colab_outputs_seed{SEED}.zip /content/drive/MyDrive/argumind_outputs/
    print("Tersalin ke Google Drive!")